<a href="https://colab.research.google.com/github/shalini-vishwakarma/AI-Product-Portfolio/blob/main/stockmarket.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Install compatible versions of `numpy==1.26.4`, `scipy==1.11.4`, `scikit-learn==1.3.2`, along with `yfinance`, `gradio`, and `plotly` to resolve dependency conflicts, and then execute the 'Institutional Grade AI Trading Suite (v2.0 Portfolio Manager)' application.

## Install specific numpy version

### Subtask:
Install `numpy==1.26.4` to ensure compatibility with other libraries.


**Reasoning**:
The subtask is to install `numpy==1.26.4`. I will use `!pip install` with the specified version and the `--quiet` flag.



In [ ]:
print("Installing numpy==1.26.4...")
!pip install numpy==1.26.4 --quiet
print("numpy==1.26.4 installed.")

Installing numpy==1.26.4...
numpy==1.26.4 installed.


## Install specific scipy version

### Subtask:
Install `scipy==1.11.4` to ensure compatibility with other libraries.

**Reasoning**:
The next step is to install `scipy==1.11.4` to resolve dependency conflicts. I will use `!pip install` with the specified version and the `--quiet` flag.



**Reasoning**:
The next step is to install `scikit-learn==1.3.2` to resolve dependency conflicts. I will use `!pip install` with the specified version and the `--quiet` flag.



In [ ]:
print("Installing scikit-learn==1.3.2...")
!pip install scikit-learn==1.3.2 --quiet
print("scikit-learn==1.3.2 installed.")

Installing scikit-learn==1.3.2...
scikit-learn==1.3.2 installed.


**Reasoning**:
The current subtask to install scikit-learn==1.3.2 has been completed. The remaining dependencies (`yfinance`, `gradio`, `plotly`) need to be installed. I will install these using `!pip install` with the `--quiet` flag.



**Reasoning**:
All specified dependencies have been installed. The next step is to execute the 'Institutional Grade AI Trading Suite (v2.0 Portfolio Manager)' application code to verify that the dependency conflicts are resolved and the application runs as expected.



**Reasoning**:
The previous execution failed because the cell `4083dc7d` contained installation commands that conflicted with the pre-installed, compatible versions of numpy, scipy, and scikit-learn. The installation commands within the application code itself caused a re-installation of incompatible versions, leading to the `ModuleNotFoundError`. I will remove the installation commands from the provided application code block, as these dependencies have already been successfully installed in the preceding steps. This ensures that the application uses the correctly pinned versions.



In [ ]:
# ==========================================
# 🎯 INSTITUTIONAL GRADE AI TRADING SUITE (v2.0 Portfolio Manager)
# ==========================================

# 2. Imports
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import gradio as gr
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score
import warnings

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 📊 1. FEATURE ENGINEERING
# ---------------------------------------------------------
def prepare_data(ticker):
    try:
        stock = yf.Ticker(ticker)
        df = stock.history(period="2y")

        if len(df) < 200:
            return None, None, "Insufficient Data (Need > 200 days)"

        # Technical Indicators
        df['SMA_50'] = df['Close'].rolling(window=50).mean()
        df['SMA_200'] = df['Close'].rolling(window=200).mean()
        df['Above_SMA200'] = np.where(df['Close'] > df['SMA_200'], 1, 0)

        # RSI
        delta = df['Close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        df['RSI'] = 100 - (100 / (1 + rs))

        # ATR (Volatility)
        high_low = df['High'] - df['Low']
        high_close = np.abs(df['High'] - df['Close'].shift())
        low_close = np.abs(df['Low'] - df['Close'].shift())
        ranges = pd.concat([high_low, high_close, low_close], axis=1)
        true_range = np.max(ranges, axis=1)
        df['ATR'] = true_range.rolling(window=14).mean()

        # Volume
        df['Rel_Vol'] = df['Volume'] / df['Volume'].rolling(window=20).mean()

        # Target (Did price go UP tomorrow?)
        df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

        return df.dropna(), stock.info, None

    except Exception as e:
        return None, None, str(e)

# ---------------------------------------------------------
# 🧠 2. AI ENGINE
# ---------------------------------------------------------
def train_and_predict(df):
    predictors = ['RSI', 'Rel_Vol', 'Above_SMA200', 'ATR']
    train_size = int(len(df) * 0.8)
    train = df.iloc[:train_size]
    test = df.iloc[train_size:]

    model = RandomForestClassifier(n_estimators=100, min_samples_split=10, random_state=42)
    model.fit(train[predictors], train['Target'])

    preds = model.predict(test[predictors])
    accuracy = precision_score(test['Target'], preds, zero_division=0)

    last_row = df.iloc[[-1]][predictors]
    prediction_prob = model.predict_proba(last_row)[0][1]

    return prediction_prob, accuracy

# ---------------------------------------------------------
# 💼 3. PORTFOLIO MANAGER LOGIC (The New Feature)
# ---------------------------------------------------------
def analyze_existing_position(buy_price, current_price, is_bullish, atr):
    """
    Decides whether to Hold, Sell, or Add based on PnL + AI Trend.
    """
    if buy_price is None or buy_price <= 0:
        return None # User doesn't own it

    pnl_pct = ((current_price - buy_price) / buy_price) * 100
    pnl_color = "green" if pnl_pct >= 0 else "red"

    # Logic Matrix
    action = ""
    reason = ""
    badge_color = ""

    # SCENARIO 1: We are in PROFIT
    if pnl_pct > 0:
        if is_bullish:
            action = "HOLD 🗒"
            reason = "Trend is Up + You are Green. Let winners run."
            badge_color = "#27ae60" # Green
        else:
            action = "SELL / LOCK PROFIT 💰"
            reason = "Trend has reversed Bearish. Protect your gains."
            badge_color = "#f39c12" # Orange

    # SCENARIO 2: We are in LOSS
    else:
        if is_bullish:
            action = "HOLD (Wait) ⏳"
            reason = "You are down, but AI predicts a bounce. Wait for recovery."
            badge_color = "#3498db" # Blue
        else:
            action = "CUT LOSS ✂️"
            reason = "Trend is Down + You are Red. Don't hold a losing bag."
            badge_color = "#c0392b" # Red

    # HTML for the Portfolio Box
    html_block = f"""
    <div style="margin-top: 20px; background: #fff; border: 2px solid {badge_color}; border-radius: 8px; overflow: hidden;">
        <div style="background: {badge_color}; color: white; padding: 10px 15px; font-weight: bold; font-size: 16px;">
            💼 POSITION ADVICE: {action}
        </div>
        <div style="padding: 15px; display: flex; justify-content: space-between; align-items: center;">
            <div>
                <div style="font-size: 12px; color: #7f8c8d;">Your P&L</div>
                <div style="font-size: 20px; font-weight: bold; color: {pnl_color};">
                    {pnl_pct:.2f}%
                </div>
            </div>
            <div style="text-align: right; max-width: 70%;">
                <div style="font-size: 14px; color: #333;">{reason}</div>
            </div>
        </div>
    </div>
    """
    return html_block

# ---------------------------------------------------------
# 🚀 MAIN APP LOGIC
# ---------------------------------------------------------
def run_analysis(ticker, buy_price_input):
    ticker = ticker.strip().upper()
    if not ticker: return "Please enter a ticker", None

    # Handle optional buy price
    try:
        buy_price = float(buy_price_input) if buy_price_input else 0
    except:
        buy_price = 0

    # 1. Get Data
    df, info, err = prepare_data(ticker)
    if err: return f"Error: {err}", None

    current_price = df['Close'].iloc[-1]
    current_atr = df['ATR'].iloc[-1]

    # 2. Run AI Model
    prob_up, model_accuracy = train_and_predict(df)

    # 3. Determine Market Bias
    if prob_up > 0.55:
        direction_up = True
        signal_strength = "BULLISH 🟢"
        confidence = prob_up
    elif prob_up < 0.45:
        direction_up = False
        signal_strength = "BEARISH 🔴"
        confidence = 1 - prob_up
    else:
        direction_up = True
        signal_strength = "NEUTRAL ⚪"
        confidence = 0.5

    # 4. Calculate New Trade Levels (For new buyers)
    sl_dist = current_atr * 2
    tp_dist = sl_dist * 2
    if direction_up:
        sl = current_price - sl_dist
        tp = current_price + tp_dist
    else:
        sl = current_price + sl_dist
        tp = current_price - tp_dist

    # 5. Generate Portfolio Advice (If user owns it)
    portfolio_html = ""
    if buy_price > 0:
        portfolio_html = analyze_existing_position(buy_price, current_price, direction_up, current_atr)

    # 6. Graham Number
    try:
        eps = info.get('trailingEps', 0)
        bvps = info.get('bookValue', 0)
        graham_num = (22.5 * eps * bvps) ** 0.5 if (eps > 0 and bvps > 0) else 0
    except:
        graham_num = 0

    # 7. Generate Chart
    fig = go.Figure()
    fig.add_trace(go.Candlestick(x=df.index, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'], name="Price"))
    fig.add_hline(y=sl, line_dash="dash", line_color="red", annotation_text="New Trade SL")
    fig.add_hline(y=tp, line_dash="dash", line_color="green", annotation_text="New Trade TP")
    # If user has a position, show their buy line
    if buy_price > 0:
        fig.add_hline(y=buy_price, line_width=2, line_color="blue", annotation_text="YOUR ENTRY")

    fig.update_layout(title=f"Analysis: {ticker}", template="plotly_white", height=500, xaxis_rangeslider_visible=False)

    # 8. HTML Report
    color = "#27ae60" if direction_up else "#c0392b"
    if "NEUTRAL" in signal_strength: color = "#7f8c8d"

    html = f"""
    <div style="font-family: 'Segoe UI', sans-serif; max-width: 800px; margin: auto; background: #fff; padding: 25px; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.1);">

        <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #f0f0f0; padding-bottom: 15px;">
            <div>
                <h2 style="margin: 0; color: #2c3e50; font-size: 28px;">{ticker}</h2>
                <span style="font-size: 18px; color: #7f8c8d;">Current: <b>{current_price:.2f}</b></span>
            </div>
            <div style="text-align: right;">
                <div style="background: {color}; color: white; padding: 8px 20px; border-radius: 30px; font-weight: bold;">{signal_strength}</div>
                <div style="font-size: 12px; color: #7f8c8d; margin-top: 5px;">AI Conf: {confidence*100:.1f}%</div>
            </div>
        </div>

        <!-- PORTFOLIO ADVICE INJECTED HERE -->
        {portfolio_html if portfolio_html else ""}

        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-top: 25px;">
            <div style="background: #fdfefe; padding: 15px; border: 1px solid #eee; border-radius: 8px;">
                <div style="font-size: 12px; color: #7f8c8d; font-weight: bold;">NEW TRADE TARGET</div>
                <div style="font-size: 22px; color: #27ae60; font-weight: bold;">{tp:.2f}</div>
            </div>
            <div style="background: #fdfefe; padding: 15px; border: 1px solid #eee; border-radius: 8px;">
                <div style="font-size: 12px; color: #7f8c8d; font-weight: bold;">NEW TRADE STOP LOSS</div>
                <div style="font-size: 22px; color: #c0392b; font-weight: bold;">{sl:.2f}</div>
            </div>
        </div>

        <div style="margin-top: 20px; padding: 15px; background: #f8f9fa; border-radius: 8px; font-size: 13px;">
            <b>🧠 AI Diagnostics:</b> Model Accuracy {model_accuracy*100:.1f}% | RSI {df['RSI'].iloc[-1]:.1f} | Graham Value {graham_num:.2f}
        </div>
    </div>
    """

    return html, fig

# ---------------------------------------------------------
# UI
# ---------------------------------------------------------
with gr.Blocks(title="Institutional AI Trader") as demo:
    gr.Markdown("# 🎯 Institutional AI: Portfolio Manager")

    with gr.Row():
        with gr.Column(scale=1):
            txt_input = gr.Textbox(label="Ticker", placeholder="e.g. NVDA")
            buy_input = gr.Number(label="Your Avg Buy Price (Optional)", value=0)
            btn_run = gr.Button("Analyze Position", variant="primary")
        with gr.Column(scale=3):
            out_html = gr.HTML(label="Report")

    out_plot = gr.Plot(label="Chart")

    btn_run.click(run_analysis, inputs=[txt_input, buy_input], outputs=[out_html, out_plot])

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4024ea8a9fadab03bd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
